# Chapitre 6 — Statistiques et agrégations

## Le problème

Une table de 240 commandes ne se lit pas ligne à ligne. On veut la **résumer** : chiffre d'affaires total, panier moyen, nombre de clients distincts, répartition par canal. Une agrégation réduit un ensemble de valeurs à une seule : c'est ce que font `SUM`, `AVG`, `COUNT` en SQL.

Ce chapitre traite les agrégations sur toute une table. Le chapitre 7 les combinera avec des regroupements (`GROUP BY`).

In [ ]:
import numpy as np
import pandas as pd

commandes = pd.read_csv("../donnees/demo/commandes.csv", parse_dates=["date_commande"])
produits = pd.read_csv("../donnees/demo/produits.csv")

# prix unitaire de chaque commande : on cherche le prix du produit dans la table produits
prix_par_produit = produits.set_index("produit_id")["prix"]
commandes["prix_unitaire"] = commandes["produit_id"].map(prix_par_produit)
commandes["montant"] = commandes["quantite"] * commandes["prix_unitaire"]
commandes.head()

`map` remplace chaque identifiant de produit par la valeur correspondante dans la Series `prix_par_produit`, utilisée comme un dictionnaire. Le chapitre 8 présentera la jointure, qui fait la même chose de façon plus générale.

## 1. Les agrégations de base

In [ ]:
montant = commandes["montant"]
print("somme      :", montant.sum().round(2))
print("moyenne    :", montant.mean().round(2))
print("médiane    :", montant.median())
print("écart-type :", montant.std().round(2))
print("variance   :", montant.var().round(2))
print("min / max  :", montant.min(), "/", montant.max())
print("nombre     :", montant.count())

| Méthode | SQL | Remarque |
|---|---|---|
| `sum()` | `SUM` | |
| `mean()` | `AVG` | |
| `median()` | — | robuste aux valeurs extrêmes |
| `std()`, `var()` | `STDDEV`, `VARIANCE` | **échantillon** (division par n − 1) par défaut, `ddof=0` pour la population |
| `min()`, `max()` | `MIN`, `MAX` | |
| `count()` | `COUNT(col)` | ne compte **pas** les valeurs manquantes |
| `size` (attribut) | `COUNT(*)` | compte toutes les lignes |
| `nunique()` | `COUNT(DISTINCT col)` | |
| `quantile(q)` | `PERCENTILE_CONT` | |
| `mode()` | — | la ou les valeurs les plus fréquentes |
| `idxmax()`, `idxmin()` | — | l'**étiquette** de la valeur extrême |

In [ ]:
montant.quantile([0.25, 0.5, 0.75, 0.9])

In [ ]:
commandes["canal"].mode()

In [ ]:
commandes.loc[montant.idxmax()]          # la commande au montant le plus élevé

### Les valeurs manquantes sont ignorées

In [ ]:
notes = pd.Series([12, 15, np.nan, 9])
notes.mean(), notes.sum(), notes.count(), notes.size

> **Prédiction.** La moyenne vaut 12, pas 9. Pourquoi ? Que donnerait `notes.mean(skipna=False)` ?

In [ ]:
notes.mean(skipna=False)

Par défaut, `skipna=True` : les agrégations ignorent les `NaN`. La moyenne porte donc sur 3 valeurs, pas 4. C'est généralement souhaitable, mais il faut le savoir : une colonne aux trois quarts vide aura une moyenne calculée sur un quart des lignes.

## 2. Le paramètre `axis`

Sur un DataFrame, une agrégation s'applique par défaut **à chaque colonne** (`axis=0` : on réduit les lignes). Avec `axis=1`, elle s'applique à chaque ligne. C'est la même convention qu'en NumPy.

In [ ]:
ventes = pd.DataFrame({"web": [10, 4, 7], "magasin": [3, 8, 2], "telephone": [1, 0, 2]},
                      index=["P01", "P02", "P03"])
ventes

In [ ]:
ventes.sum()              # par colonne : total par canal

In [ ]:
ventes.sum(axis=1)        # par ligne : total par produit

In [ ]:
ventes.sum().sum()        # total général

### Colonnes de types mélangés : `numeric_only`

> **Trouver l'erreur.** Pourquoi la cellule suivante échoue-t-elle ?

In [ ]:
commandes.mean()

Pandas ne sait pas calculer la moyenne d'un texte. En Pandas 3, le type `str` refuse ces opérations au lieu de produire un résultat absurde. On restreint le calcul aux colonnes numériques :

In [ ]:
commandes.mean(numeric_only=True)

La moyenne de `commande_id` ou de `client_id` n'a aucun sens : une colonne codée en nombres n'est pas forcément une mesure. Mieux vaut sélectionner explicitement les colonnes concernées.

## 3. `agg` : plusieurs agrégations à la fois

`agg` reçoit une liste de fonctions, un dictionnaire par colonne, ou des agrégations nommées.

In [ ]:
commandes["montant"].agg(["sum", "mean", "median", "max"])

In [ ]:
commandes.agg({"montant": ["sum", "mean"], "quantite": ["sum", "max"], "client_id": "nunique"})

Les **agrégations nommées** donnent directement un nom à chaque résultat, comme un alias `AS` en SQL :

```sql
SELECT SUM(montant) AS ca_total, AVG(montant) AS panier_moyen, COUNT(DISTINCT client_id) AS nb_clients
FROM commandes;
```

In [ ]:
commandes.agg(
    ca_total=("montant", "sum"),
    panier_moyen=("montant", "mean"),
    nb_clients=("client_id", "nunique"),
)

On peut aussi passer une fonction Python à `agg`, par exemple l'étendue (max − min) :

In [ ]:
commandes["montant"].agg(lambda x: x.max() - x.min())

## 4. Compter les valeurs : `value_counts`

`value_counts` compte les occurrences de chaque valeur, triées par fréquence décroissante. C'est l'outil de base pour décrire une variable catégorielle.

In [ ]:
commandes["canal"].value_counts()

In [ ]:
commandes["canal"].value_counts(normalize=True).round(3)      # proportions

Sur un DataFrame, `value_counts` compte les **combinaisons** de colonnes :

In [ ]:
commandes.value_counts(["canal", "statut"]).head()

Par défaut, les valeurs manquantes ne sont pas comptées. `dropna=False` les fait apparaître :

In [ ]:
jo = pd.read_csv("../donnees/bstorm/Exercices_Pandas_athlete_events.csv")
jo["Medal"].value_counts(dropna=False)

`nunique` et `unique` complètent l'inventaire :

In [ ]:
commandes["client_id"].nunique(), commandes["produit_id"].unique()[:5]

## 5. Une question de performance : `np.sum` contre `sum`

Les méthodes Pandas et les fonctions NumPy s'exécutent en code compilé. La fonction `sum` intégrée à Python parcourt les valeurs une par une.

> **Comparaison.** Quel écart de temps attendez-vous entre ces trois écritures ?

In [ ]:
grand = pd.Series(np.random.default_rng(0).random(1_000_000))
%timeit sum(grand)
%timeit np.sum(grand)
%timeit grand.sum()

L'écart se compte en ordres de grandeur. Sur une Series, on utilise toujours la méthode (`s.sum()`) ou la fonction NumPy, jamais la fonction intégrée.

## 6. Relations entre variables : un premier regard

En préparation de données pour un modèle, on regarde aussi si deux variables numériques évoluent ensemble. `corr` calcule le coefficient de corrélation linéaire, entre −1 et 1.

In [ ]:
jo[["Age", "Height", "Weight"]].corr().round(2)

La taille et le poids sont fortement liés (0,8) ; l'âge est presque indépendant des deux. Une forte corrélation entre variables explicatives est une information utile avant de choisir les variables d'un modèle.

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| Agréger un DataFrame aux colonnes mixtes | `TypeError` | `numeric_only=True` ou sélectionner les colonnes |
| `count()` pour compter des lignes | résultat plus petit que prévu | `count()` ignore les `NaN` ; `len(df)` ou `size` comptent tout |
| Moyenne d'une colonne très incomplète | représentativité douteuse | regarder `count()` à côté de `mean()` |
| `std()` Pandas ≠ `np.std()` | écart dans les résultats | Pandas divise par n − 1, NumPy par n (`ddof`) |
| Moyenne d'un identifiant | résultat sans signification | ne pas agréger les codes |
| `sum()` intégrée de Python | lenteur | `s.sum()` |

## Exercices

- Exercice Bstorm **JO**, questions **18 et 19** (si ce n'est pas déjà fait), et question **29** en préparation du chapitre 7.
- Renforcement : drill **D06 US Baby Names**.

## À retenir

- `sum`, `mean`, `median`, `std`, `min`, `max`, `count`, `nunique`, `quantile`, `idxmax`.
- Les `NaN` sont ignorés par défaut (`skipna=True`).
- `axis=0` réduit les lignes (un résultat par colonne), `axis=1` réduit les colonnes.
- `agg` combine plusieurs agrégations ; les agrégations nommées jouent le rôle de `AS`.
- `value_counts` décrit une variable catégorielle, `normalize=True` donne des proportions.